# LTX-2.5 22B Distilled · Kaggle T4 x2 一键册

第三条线，结构和前两册一致：**配置 → 环境 → 权重 → 启动**。

| 项 | 说明 |
|---|---|
| 加速器 | **GPU T4 x2** |
| Internet | 打开 |
| 引擎 | `run_ltx25.py`（自注册 `int8_convrot` 算子，不依赖 Wan2GP 的 `wgp` UI 模块） |
| 权重 | `/kaggle/tmp/ltx25_models`，引擎硬性检查 DiT + Gemma 两个文件 |
| Wan2GP | 默认 `b8b18f81`（2026-10-01）；若报 `wgp` / settings 相关错误，改 Step 0 的 `WAN2GP_COMMIT` 重跑 |

**⚠️ 权重是唯一需要你自己确认的点**：仓库里没有 LTX-2.5 的下载脚本，下面只按引擎要求的文件名去找。
最省事的做法是把 2.5 权重挂成 Kaggle Dataset（文件名保持原样），Step 2 会自动从 `/kaggle/input` 捡起；
或者把 Step 2 里的 `LTX25_REPO` / `LTX25_REV` 填成你手上的仓库。

**Secret**：优先 Kaggle Secrets，其次 Step 0 硬编码；`HF_TOKEN` 私有仓库才需要。


In [ ]:
# ============ 0. 配置（只改这一段） ============
import os, sys, json, math, time, glob, shutil, subprocess

IS_KAGGLE = os.path.isdir("/kaggle")
WORK = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./kaggle_working")
TMP  = "/kaggle/tmp"     if IS_KAGGLE else os.path.abspath("./kaggle_tmp")
os.makedirs(WORK, exist_ok=True)
os.makedirs(TMP, exist_ok=True)

# ---- Secrets：先读 Kaggle Secrets，取不到就用下面的硬编码值 ----
HARDCODED_SECRETS = {
    "HF_TOKEN": "",            # 私有/限流仓库才需要
    "KAGGLE_USERNAME": "",
    "KAGGLE_KEY": "",
    "CF_PRIVATE_HOST": "",
    "CF_TUNNEL_TOKEN": "",
}

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            print(f"🔑 {name}: 来自 Kaggle Secrets")
            return v
    except Exception:
        pass
    v = HARDCODED_SECRETS.get(name, "")
    print(f"🔑 {name}: {'使用硬编码值' if v else '未设置（可选）'}")
    return v

HF_TOKEN = secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "CF_PRIVATE_HOST", "CF_TUNNEL_TOKEN"):
    _v = secret(_k)
    if _v:
        os.environ[_k] = _v

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---- 仓库 / Wan2GP ----
REPO_URL   = "https://github.com/milleniummike-del/kaggle-minimax.git"
REPO_DIR   = os.path.join(WORK, "kaggle-minimax")
WAN2GP_DIR = os.path.join(WORK, "Wan2GP")
WAN2GP_URL = "https://github.com/DeepBeepMeep/Wan2GP.git"
WAN2GP_COMMIT = "b8b18f8114e432eea8f3d7e853a51dd91fa99571"

print("=" * 70)
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"], text=True).strip()
    for _i, _l in enumerate(smi.splitlines()):
        print(f"✅ GPU {_i}: {_l}")
    if len(smi.splitlines()) < 2:
        print("⚠️ 本册需要 2 块 GPU：Settings → Accelerator → GPU T4 x2")
except Exception:
    print("⚠️ 没检测到 GPU（下载/环境步骤仍可跑）")
print(f"💾 {TMP} 可用 {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
print("=" * 70)


In [ ]:
# ============ 1. 环境：拉仓库 + 固定 Wan2GP + 装依赖 ============
import os, sys, subprocess

def _run(cmd, **kw):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

# 1a. 引擎仓库（每册都从这里取 frontend.py / run_ltx*.py）
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    _run(["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", "HEAD"])
    _run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    _run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR])
print("✅ 仓库就绪:", REPO_DIR)

# 1b. Wan2GP 固定到引擎适配的 commit（换版本会改变算子/精度路径）
if not os.path.isdir(os.path.join(WAN2GP_DIR, ".git")):
    shutil.rmtree(WAN2GP_DIR, ignore_errors=True)
    os.makedirs(WAN2GP_DIR, exist_ok=True)
    _run(["git", "-C", WAN2GP_DIR, "init", "-q"])
    _run(["git", "-C", WAN2GP_DIR, "remote", "add", "origin", WAN2GP_URL])
_run(["git", "-C", WAN2GP_DIR, "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT])
_run(["git", "-C", WAN2GP_DIR, "reset", "-q", "--hard", "FETCH_HEAD"])
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]}")

# 1d. 依赖：Wan2GP requirements（把已下线的 onnxruntime nightly 换成正式版）+ 2.5 的 INT8 组合。
#     注意：run_ltx25.py 启动时会自己给 Wan2GP 打 7 处源码补丁，所以这里不重复打 2.3 那套。
print("📦 安装 Wan2GP requirements ...")
with open(os.path.join(WAN2GP_DIR, "requirements.txt")) as _f:
    _reqs = [("onnxruntime-gpu>=1.22.0" if _l.startswith("onnxruntime-gpu==") and "dev" in _l
              else _l.rstrip("\n")) for _l in _f]
_req = os.path.join(WORK, "wan2gp_requirements.txt")
open(_req, "w").write("\n".join(_reqs) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                "--timeout", "120", "--retries", "5", "-r", _req], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
                "--timeout", "120", "--retries", "5",
                "mmgp==3.8.2", "comfy-kitchen==0.2.35", "gradio>=5.0.0",
                "fastapi>=0.100.0", "uvicorn>=0.20.0", "python-multipart",
                "gguf", "soundfile", "einops", "sentencepiece", "hf_transfer", "accelerate"],
               check=True)
print("✅ LTX-2.5 环境就绪（Wan2GP 源码补丁由 run_ltx25.py 启动时自己打）")


In [ ]:
# ============ 2. 权重 → /kaggle/tmp/ltx25_models ============
# ⚠️ 仓库没有 LTX-2.5 的下载脚本。引擎只硬性检查下面两个文件；其余（VAE / audio VAE /
#    vocoder）会按文件名前缀从 /kaggle/input 里自动捡起，所以最省事的做法是把 2.5 权重
#    挂成 Dataset（文件名保持原样），或填好下面的 LTX25_REPO / LTX25_REV。
from huggingface_hub import hf_hub_download

LTX25_REPO = "DeepBeepMeep/LTX-2"   # ← 换成本地/别处的 2.5 仓库
LTX25_REV  = None                   # ← 需要可复现就固定 revision

MODELS_DIR = os.path.join(TMP, "ltx25_models")
GEMMA_DIR  = os.path.join(MODELS_DIR, "gemma4-12b-ltx-v1")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(GEMMA_DIR, exist_ok=True)

REQUIRED = [
    ("ltx-2.5-22b-distilled_diffusion_model_int8_convrot.safetensors", MODELS_DIR),
    ("gemma4-12b-ltx-v1/gemma4-12b-ltx-v1_int8_convrot.safetensors",   MODELS_DIR),
]
EXTRA_HINTS = ("ltx-2.5", "gemma4-12b-ltx-v1")

_INDEX = {}
if os.path.isdir("/kaggle/input"):
    for _root, _, _files in os.walk("/kaggle/input"):
        for _f in _files:
            _INDEX.setdefault(_f, os.path.join(_root, _f))

def pull(repo, filename, dest_dir):
    dest = os.path.join(dest_dir, filename)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"  ✓ 已存在 {filename}")
        return True
    src = _INDEX.get(os.path.basename(filename))
    if src:
        os.symlink(src, dest)
        print(f"  ⚡ 复用 Dataset {filename}")
        return True
    try:
        print(f"  ⬇️ {filename}")
        hf_hub_download(repo_id=repo, filename=filename, revision=LTX25_REV,
                        local_dir=dest_dir, token=os.environ.get("HF_TOKEN") or None)
        return True
    except Exception as e:
        print(f"  ❌ {filename}: {type(e).__name__}: {e}")
        return False

_ok = all(pull(LTX25_REPO, f, d) for f, d in REQUIRED)

# 可选：2.5 的 VAE / 音频链，仓库里没有清单，按命名惯例试；缺了不影响启动，
# 引擎加载时会告诉你具体缺哪个文件。
OPTIONAL = [
    "ltx-2.5-22b_vae.safetensors",
    "ltx-2.5-22b_audio_vae.safetensors",
    "ltx-2.5-22b_vocoder.safetensors",
    "ltx-2.5-22b_text_embedding_projection.safetensors",
    "ltx-2.5-22b_embeddings_connector.safetensors",
]
for _f in OPTIONAL:
    if not os.path.exists(os.path.join(MODELS_DIR, _f)):
        pull(LTX25_REPO, _f, MODELS_DIR)

# 额外：/kaggle/input 下所有 2.5 相关文件，平铺软链进 MODELS_DIR（VAE / vocoder 等）
_adopted = 0
for _name, _src in _INDEX.items():
    if any(_h in _name for _h in EXTRA_HINTS):
        _dst = os.path.join(MODELS_DIR, _name)
        if not os.path.exists(_dst):
            try:
                os.symlink(_src, _dst)
                _adopted += 1
            except OSError:
                pass

print(f"\n{'✅' if _ok else '⚠️'} LTX-2.5 权重: 必需文件{'齐了' if _ok else '缺失'} | 从 Dataset 追加 {_adopted} 个")
if not _ok:
    print("   请把这几个文件挂到 /kaggle/input（保持文件名），或改 LTX25_REPO / LTX25_REV 后重跑本格。")


In [ ]:
# ============ 3. 启动引擎（跑完会出现 Cloudflare / Gradio 公网链接） ============
# 这一格会一直占着，日志会实时刷出来；中断内核或点停止即可关服务，重跑本格=重启。
import codecs, re, time
import psutil

ENGINE_DIR = WORK
ENGINE     = os.path.join(REPO_DIR, "run_ltx25.py")
EXTRA_ENV  = {"LTX25_MODELS_DIR": os.path.join(TMP, "ltx25_models")}

assert os.path.isfile(ENGINE), f"引擎不存在：{ENGINE}（先跑 Step 1）"
for _k, _v in list(EXTRA_ENV.items()):
    if _k.endswith("_DIR"):
        assert os.path.exists(_v), f"缺目录：{_v}（先跑 Step 2）"

# 清掉上一轮残留的引擎 / cloudflared，避免重复占显存
for _p in psutil.process_iter(["name", "cmdline"]):
    try:
        _args = _p.info["cmdline"] or []
        _name = (_p.info["name"] or "").lower()
        _hit = any(os.path.basename(a) == os.path.basename(ENGINE) for a in _args[1:3])
        if _p.pid != os.getpid() and (_hit or _name == "cloudflared"):
            _p.kill()
    except Exception:
        pass
time.sleep(1)

ENV = os.environ.copy()
ENV.update({k: str(v) for k, v in EXTRA_ENV.items()})
print(f"🚀 cd {ENGINE_DIR} && python {os.path.basename(ENGINE)}")
sys.stdout.flush()

proc = subprocess.Popen([sys.executable, ENGINE], cwd=ENGINE_DIR, env=ENV,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()
        out = "".join(parts)
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        print(f"\n❌ 引擎退出（code={rc}）。code=-9 通常是主机内存不够。")
except KeyboardInterrupt:
    proc.kill()
    proc.wait()
    print("🛑 已停止。重跑本格即可再启动。")
